In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# define catalgo and schema for this scprit
project_catalog='data_project'
bronze_table="bronze.raw_scraped_result"
silver_table="silver.clean_scraped_data"

spark.sql(f"USE CATALOG {project_catalog}")

# read data
raw_data=spark\
    .read\
    .table(f"{bronze_table}")

In [0]:
def drop_duplicate(df, column) :
    window_function=Window.partitionBy(column).orderBy(F.desc(column))
    df=df\
        .withColumn("rownumber", F.row_number().over(window_function))\
        .filter(F.col("rownumber") == 1)\
        .drop("rownumber")
    return df
    
def clean_review(df,column,separator) :
    df=df\
        .withColumn(column, F.split(F.col(column), separator)[0])\
        .withColumn(column, F.regexp_replace(',',''))\
        .withColumn(column, F.col(column).cast('int'))
    return df

def clean_rating(df,column,separator) :
    df=df\
        .withColumn(column, F.split(F.col(column), separator)[0])\
        .withColumn(column, F.col(column).cast('float'))
    return df

def clean_review(df,column,separator) :
    df=df\
        .withColumn(column, F.split(F.col(column), separator)[0])\
        .withColumn(column, F.regexp_replace(column,',',''))\
        .withColumn(column, F.col(column).cast('int'))
    return df

def standarize_phone_number(df,column) :
    df = df\
            .withColumn(column, F.when(F.col(column).startswith('08'), F.regexp_replace(column, "[^0-9]", "")).otherwise(F.col(column)))\
            .withColumn(column, F.when(F.col(column).startswith('08'), F.regexp_replace(column, "^08", "+62")).otherwise(F.col(column)))\
            .withColumn(column, F.regexp_replace(column, " ", ""))
    return df

df_result=raw_data\
        .transform(drop_duplicate, "id_lokasi")\
        .transform(clean_rating, "rating_lokasi"," ")\
        .transform(clean_review, "review_lokasi"," ")\
        .transform(standarize_phone_number, "nohp_lokasi")\
        .withColumn("nama_lokasi", F.initcap(F.col("nama_lokasi")))\
        .withColumn("wkt_scraping",  F.try_to_timestamp(F.col("wkt_scraping"), F.lit("dd/MM/yyyy HH:mm")))\
        .withColumn("cleaned_at", F.from_utc_timestamp(F.current_timestamp(), "Asia/Jakarta"))

        
df_result.show(10)

In [0]:
rename_map = {
    "id_lokasi": "location_id",
    "nama_lokasi": "location_name",
    "tag_lokasi": "location_tags",
    "url_lokasi": "location_url",
    "website_lokasi": "website",
    "alamat_lokasi": "address",
    "provinsi": "province",
    "kab_kota": "city_regency",
    "kecamatan": "district",
    "keluarahan_desa": "subdistrict_village",
    "kodepos": "postal_code",
    "latitude": "latitude",
    "longitude": "longitude",
    "nohp_lokasi": "phone_number",
    "jam_operasional": "operating_hours",
    "rating_lokasi": "rating",
    "review_lokasi": "reviews",
    "wkt_scraping": "scraped_at",
    "cleaned_at": "cleaned_at",
}

# Untuk PySpark 3.4+
df_result = df_result.withColumnsRenamed(rename_map)

In [0]:
#create temp table as reference for creating table and insert data
df_result.createOrReplaceTempView("clean_table")

#create teable if table is not exist yet
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {project_catalog}.{silver_table}
AS 
SELECT * FROM clean_table WHERE 1 = 0""")

#upsert statement
spark.sql(f"""
          MERGE INTO {project_catalog}.{silver_table} AS T
          USING clean_table AS S
          ON T.location_id = S.location_id
          WHEN MATCHED THEN
            UPDATE SET *
          WHEN NOT MATCHED THEN
            INSERT *
          """)

In [0]:
spark.sql(f"DESCRIBE HISTORY {project_catalog}.{silver_table} LIMIT 1;").show(truncate=False)